# NetCDF File Inspector

Inspect the structure and variables of NetCDF files used in PCRGlobWB calibration.

In [3]:
import xarray as xr
from pathlib import Path
from src.paths import PCR_GLOBAL_PARAMS

In [4]:
print(PCR_GLOBAL_PARAMS)

/data/shared/parameter-sets/pcrglobwb_global


In [6]:
# Specify path to NetCDF file
# Example: the coverFractionInputForest.nc referenced in test_comp_speed_basin_path.ini
nc_path = PCR_GLOBAL_PARAMS / "global_05min/landSurface/landCover/naturalTall/coverFractionInputForest.nc"

# Adjust path if not found - try alternative locations
if not nc_path.exists():
    # Search nearby
    parent = nc_path.parent
    print(f"File not found at {nc_path}")
    print(f"Searching in {parent}...")
    if parent.exists():
        nc_files = list(parent.glob("*.nc"))
        if nc_files:
            nc_path = nc_files[0]
            print(f"Using: {nc_path}")
        else:
            print(f"No .nc files in {parent}")
    else:
        print(f"Directory {parent} does not exist")

In [7]:
# Open and inspect the NetCDF file
if nc_path.exists():
    ds = xr.open_dataset(nc_path)
    print(ds)
    print("\n" + "="*60)
    print(f"\nVariables: {list(ds.data_vars)}")
    print(f"Dimensions: {dict(ds.dims)}")
    print(f"Coordinates: {list(ds.coords)}")
else:
    print(f"File still not found: {nc_path}")

KeyboardInterrupt: 

In [ ]:
# Sample the data if file was opened successfully
if nc_path.exists():
    ds = xr.open_dataset(nc_path)
    for var in ds.data_vars:
        print(f"\n{var}:")
        print(f"  Shape: {ds[var].shape}")
        print(f"  Dtype: {ds[var].dtype}")
        print(f"  Min: {ds[var].min().values}, Max: {ds[var].max().values}")

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

# FALLBACK_EXTENT for Aral region
FALLBACK_EXTENT = (54, 33, 82, 53)
lon_min, lat_min, lon_max, lat_max = FALLBACK_EXTENT

# Open the NetCDF file
if nc_path.exists():
    ds = xr.open_dataset(nc_path)
    
    # Get the data variable (usually the first one)
    var_name = list(ds.data_vars)[0]
    data = ds[var_name].values
    
    # Extract lat/lon coordinates
    lat = ds.coords['lat'].values if 'lat' in ds.coords else None
    lon = ds.coords['lon'].values if 'lon' in ds.coords else None
    
    # Create map
    fig = plt.figure(figsize=(12, 8), dpi=100)
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    
    # Plot data
    im = ax.imshow(data, extent=[lon.min(), lon.max(), lat.min(), lat.max()],
                   transform=ccrs.PlateCarree(), cmap='viridis', origin='upper')
    
    # Add features
    ax.coastlines()
    ax.borders()
    ax.gridlines(draw_labels=True)
    
    plt.colorbar(im, ax=ax, label=var_name, shrink=0.8)
    plt.title(f"{var_name} for Aral Region\n{FALLBACK_EXTENT}")
    plt.tight_layout()
    plt.show()
else:
    print(f"File not found: {nc_path}")